# 2026 COMP90042 Project

**Single-notebook system based on the teammate route, with the SBERT reranker replaced by a fine-tuned cross-encoder reranker.**

Pipeline:

1. Load provided claim/evidence files.
2. Build BM25 lexical retrieval over `evidence.json`.
3. Construct cross-encoder reranker training pairs from train claims:
   - positive pairs: `(claim, ground-truth evidence)`
   - hard negative pairs: `(claim, BM25 candidate not in ground truth)`
4. Fine-tune a lightweight Transformer cross-encoder reranker.
5. Rerank BM25 candidates with the fine-tuned cross-encoder.
6. Tune final evidence selection on dev: fixed `K` and dynamic threshold `K`.
7. Fine-tune MiniLM classifier on retrieved evidence.
8. Output `dev-predictions.json` and `test-output.json` in official format.

This notebook intentionally keeps the teammate's notebook style and does not depend on the previous `src/` project modules.

# Readme

Expected project layout:

```text
.
├── data/
│   ├── train-claims.json
│   ├── dev-claims.json
│   ├── test-claims-unlabelled.json
│   └── evidence.json
├── eval.py
└── this_notebook.ipynb
```

Main changes over the teammate SBERT route:

- Removes SBERT semantic reranking.
- Adds supervised cross-encoder reranker fine-tuning using BM25-mined hard negatives.
- Enlarges BM25 candidate pool to `500` by default.
- Sweeps final fixed `K ∈ {2,3,4,5}` and dynamic threshold `K` on dev.
- Trains the classifier on retrieved evidence rather than oracle ground-truth evidence.
- Keeps final system simple: BM25 retrieval + supervised reranker + joint MiniLM cross-encoder classifier.

For quick debugging, set `FAST_DEV_MODE = True` in the config cell. For final runs, keep it `False`.

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [ ]:
# Optional dependency installation. Safe in Colab; usually skipped locally if packages already exist.
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "bm25s": "bm25s",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "psutil": "psutil",
}

for pip_name, import_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

In [ ]:
from pathlib import Path
import json
import random
import time
import pickle
from collections import Counter

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# -------------------------
# Global config
# -------------------------
SEED = 42
FAST_DEV_MODE = False  # True = quick smoke test; False = full project run

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs_notebook_ce")
CACHE_DIR = OUTPUT_DIR / "cache"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
CACHE_DIR.mkdir(exist_ok=True, parents=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# Retrieval config
BM25_CANDIDATE_K = 500 if not FAST_DEV_MODE else 50
FIXED_K_GRID = [2, 3, 4, 5]

# Absolute CE-probability threshold. A finer grid matters because the reranker logits are not perfectly calibrated.
THRESHOLD_GRID = [round(float(x), 2) for x in np.arange(0.02, 0.52, 0.02)] + [0.60, 0.70, 0.80, 0.90]

# Relative dynamic selector: keep candidates close to the top reranker logit for that claim.
# This is more robust than an absolute threshold when logit calibration shifts between train/dev/test.
RELATIVE_LOGIT_DELTA_GRID = [0.25, 0.50, 0.75, 1.00, 1.50, 2.00, 3.00]
MIN_FINAL_K = 1
MAX_FINAL_K = 5

# Final selector policy.
# "prefer_dynamic" still tunes on dev, but if a dynamic selector is close to the best dev F,
# it chooses dynamic K because gold evidence counts vary heavily by claim.
# Set to "best_dev" for strict argmax, or to "dynamic_threshold" / "relative_logit" to force one family.
FINAL_RETRIEVAL_POLICY = "prefer_dynamic"
DYNAMIC_RETRIEVAL_TOLERANCE = 0.02

# Cross-encoder reranker config.
# Default is a lightweight open cross-encoder checkpoint. If the team wants a more conservative generic backbone,
# change this one line to "distilbert-base-uncased"; all code below remains valid.
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
RERANKER_MAX_LEN = 256
RERANKER_BATCH_SIZE = 32 if not FAST_DEV_MODE else 8
RERANKER_EVAL_BATCH_SIZE = 128 if not FAST_DEV_MODE else 16
RERANKER_EPOCHS = 3 if not FAST_DEV_MODE else 1
RERANKER_LR = 1e-5
NEGATIVES_PER_POSITIVE = 4 if not FAST_DEV_MODE else 2
HARD_NEGATIVE_POOL = min(200, BM25_CANDIDATE_K)

# Classification config
LABELS = ["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO", "DISPUTED"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}
CLASSIFIER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
MAX_SEQ_LEN = 256
TRAIN_BATCH_SIZE = 16 if not FAST_DEV_MODE else 4
EVAL_BATCH_SIZE = 32 if not FAST_DEV_MODE else 8
CLASSIFIER_EPOCHS = 8 if not FAST_DEV_MODE else 1
CLASSIFIER_LR = 1e-5
# In retrieval-conditioned classification, full inverse-frequency weighting can over-correct
# toward REFUTES/DISPUTED and hurt overall accuracy. Keep it off by default;
# set True only for an ablation.
CLASSIFIER_USE_CLASS_WEIGHTS = False
WEIGHT_DECAY = 0.01


def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


set_seed(SEED)


In [ ]:
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


train_claims = load_json(DATA_DIR / "train-claims.json")
dev_claims = load_json(DATA_DIR / "dev-claims.json")
test_claims = load_json(DATA_DIR / "test-claims-unlabelled.json")
evidence = load_json(DATA_DIR / "evidence.json")

if FAST_DEV_MODE:
    train_claims = dict(list(train_claims.items())[:80])
    dev_claims = dict(list(dev_claims.items())[:30])
    test_claims = dict(list(test_claims.items())[:30])

print(f"train:    {len(train_claims)}")
print(f"dev:      {len(dev_claims)}")
print(f"test:     {len(test_claims)}")
print(f"evidence: {len(evidence)}")

In [ ]:
# Lightweight EDA used to justify later choices.
label_dist = Counter(c["claim_label"] for c in train_claims.values())
print("Train label distribution:")
for lbl in LABELS:
    cnt = label_dist[lbl]
    print(f"  {lbl:20s} {cnt:5d} ({cnt / len(train_claims):6.2%})")

gt_counts = [len(c["evidences"]) for c in train_claims.values()]
print("\nGround-truth evidence count per train claim:")
print("  min=", min(gt_counts), "max=", max(gt_counts), "mean=", round(float(np.mean(gt_counts)), 3))
print("  exact counts:", sorted(Counter(gt_counts).items()))

In [ ]:
# Official-style metrics. These mirror eval.py's logic and let us tune inside the notebook.
def evidence_f1_for_claim(pred_eids, gold_eids):
    pred_eids = list(pred_eids)
    gold_eids = list(gold_eids)
    if len(pred_eids) == 0:
        return 0.0
    pred_set = set(pred_eids)
    correct = sum(1 for eid in gold_eids if eid in pred_set)
    if correct == 0:
        return 0.0
    precision = correct / len(pred_eids)
    recall = correct / len(gold_eids)
    return 2 * precision * recall / (precision + recall)


def evaluate_submission(predictions, gold_claims, verbose=True):
    f_scores = []
    correct_labels = 0
    total = 0
    for cid, gold in gold_claims.items():
        pred = predictions[cid]
        f_scores.append(evidence_f1_for_claim(pred["evidences"], gold["evidences"]))
        correct_labels += int(pred["claim_label"] == gold["claim_label"])
        total += 1
    F = float(np.mean(f_scores))
    A = correct_labels / total
    H = 0.0 if (F + A) == 0 else 2 * F * A / (F + A)
    if verbose:
        print(f"Evidence Retrieval F-score (F)    = {F:.6f}")
        print(f"Claim Classification Accuracy (A) = {A:.6f}")
        print(f"Harmonic Mean of F and A          = {H:.6f}")
    return {"F": F, "A": A, "H": H}


def evaluate_retrieval_only(retrieval, gold_claims):
    return float(
        np.mean(
            [
                evidence_f1_for_claim(retrieval[cid], claim["evidences"])
                for cid, claim in gold_claims.items()
            ]
        )
    )


def majority_label(claims):
    return Counter(c["claim_label"] for c in claims.values()).most_common(1)[0][0]


def validate_retrieval_coverage(claims_dict, retrieval, split_name="split"):
    """Fail early with a helpful message if retrieval is incomplete or empty."""
    missing_cids = [cid for cid in claims_dict if cid not in retrieval]
    if missing_cids:
        raise KeyError(f"{split_name}: retrieval missing {len(missing_cids)} claim ids, e.g. {missing_cids[:3]}")

    empty_cids = [cid for cid in claims_dict if len(retrieval[cid]) == 0]
    if empty_cids:
        raise ValueError(f"{split_name}: retrieval has empty evidence lists, e.g. {empty_cids[:3]}")

    bad_eids = []
    for cid in claims_dict:
        for eid in retrieval[cid]:
            if eid not in evidence:
                bad_eids.append((cid, eid))
                if len(bad_eids) >= 3:
                    break
        if len(bad_eids) >= 3:
            break
    if bad_eids:
        raise KeyError(f"{split_name}: retrieval contains unknown evidence ids, e.g. {bad_eids}")


def build_predictions(claims, retrieval, label_predictions=None, default_label=None):
    if label_predictions is None:
        assert default_label is not None
        label_predictions = {cid: default_label for cid in claims.keys()}
    out = {}
    fallback_eid = next(iter(evidence.keys()))
    for cid in claims.keys():
        eids = list(retrieval.get(cid, []))
        if len(eids) == 0:
            # Assignment requires at least one evidence. This fallback should rarely trigger.
            eids = [fallback_eid]
        out[cid] = {"claim_label": label_predictions[cid], "evidences": eids}
    return out


def write_predictions(predictions, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    # ensure_ascii=True keeps the file readable even when Windows uses a non-UTF-8 default encoding.
    with open(path, "w", encoding="utf-8") as f:
        json.dump(predictions, f, indent=2, ensure_ascii=True)
    print("Wrote", path)


## BM25 lexical retrieval

BM25 is the first-stage retriever over the full evidence corpus. The cross-encoder reranker only reranks BM25 candidates, so the candidate pool must be large enough. The default pool is `500`; `50` is only for smoke testing.

In [ ]:
import bm25s

# bm25s may print "resource module not available on Windows". It is harmless.
evidence_ids = list(evidence.keys())
evidence_texts = [evidence[eid] for eid in evidence_ids]

print(f"Tokenizing {len(evidence_texts):,} evidence passages...")
t0 = time.time()
corpus_tokens = bm25s.tokenize(evidence_texts, stopwords="en", stemmer=None)
print(f"Tokenization time: {time.time() - t0:.1f}s")

print("Building BM25 index...")
t0 = time.time()
bm25_retriever = bm25s.BM25()
bm25_retriever.index(corpus_tokens)
print(f"BM25 index time: {time.time() - t0:.1f}s")

In [ ]:
def bm25_retrieve_with_scores(claims_dict, k=BM25_CANDIDATE_K):
    cids = list(claims_dict.keys())
    queries = [claims_dict[cid]["claim_text"] for cid in cids]
    query_tokens = bm25s.tokenize(queries, stopwords="en", stemmer=None)
    results, scores = bm25_retriever.retrieve(query_tokens, k=k)

    out = {}
    for i, cid in enumerate(cids):
        pairs = []
        for j, score in zip(results[i], scores[i]):
            eid = evidence_ids[int(j)]
            pairs.append((eid, float(score)))
        out[cid] = pairs
    return out


def strip_scores(candidate_cache, k):
    return {cid: [eid for eid, _ in pairs[:k]] for cid, pairs in candidate_cache.items()}


def compute_or_load_bm25_candidates(claims_dict, split_name, k=BM25_CANDIDATE_K):
    cache_file = CACHE_DIR / f"{split_name}_bm25_top{k}.pkl"
    if cache_file.exists():
        print("Loading cached BM25 candidates:", cache_file)
        with open(cache_file, "rb") as f:
            return pickle.load(f)
    print(f"Computing BM25 candidates for {split_name}...")
    candidates = bm25_retrieve_with_scores(claims_dict, k=k)
    with open(cache_file, "wb") as f:
        pickle.dump(candidates, f)
    return candidates

In [ ]:
print(f"Retrieving BM25 top-{BM25_CANDIDATE_K} for dev...")
t0 = time.time()
dev_bm25_candidates = compute_or_load_bm25_candidates(dev_claims, "dev", BM25_CANDIDATE_K)
print(f"BM25 dev retrieval time: {time.time() - t0:.1f}s")

rows = []
for k in [5, 10, 20, 50, 100, 200, 500]:
    if k <= BM25_CANDIDATE_K:
        retr = strip_scores(dev_bm25_candidates, k)
        rows.append({"method": "BM25", "k": k, "retrieval_F": evaluate_retrieval_only(retr, dev_claims)})
pd.DataFrame(rows)

## Fine-tuned cross-encoder reranker

The teammate notebook used SBERT bi-encoder semantic similarity. This version replaces it with a supervised cross-encoder reranker:

```text
[CLS] claim [SEP] evidence [SEP] -> Transformer -> relevance logit
```

Training data is built from the provided train set:

- Positive examples are ground-truth evidence passages.
- Hard negatives are BM25 candidates that are not in the ground-truth evidence set.

This uses task supervision and should be a stronger reranker than zero-shot SBERT similarity.

In [ ]:
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

train_bm25_candidates = compute_or_load_bm25_candidates(train_claims, "train", BM25_CANDIDATE_K)


# dev candidates already computed above.


def build_reranker_examples(claims_dict, bm25_candidates, negatives_per_positive=NEGATIVES_PER_POSITIVE):
    examples = []
    rng = random.Random(SEED)

    for cid, claim in claims_dict.items():
        gold = [eid for eid in claim["evidences"] if eid in evidence]
        gold_set = set(gold)

        # Positives: each annotated evidence.
        for eid in gold:
            examples.append({"cid": cid, "eid": eid, "label": 1.0})

        # Hard negatives: sample from BM25 top-500 non-gold candidates.
        # Do not always take the very top non-gold candidates: they are often noisy negatives.
        candidate_negs = []
        seen = set()
        for eid, _ in bm25_candidates[cid][:BM25_CANDIDATE_K]:
            if eid in gold_set or eid in seen or eid not in evidence:
                continue
            candidate_negs.append(eid)
            seen.add(eid)

        needed = negatives_per_positive * max(1, len(gold))
        if len(candidate_negs) > needed:
            candidate_negs = rng.sample(candidate_negs, needed)

        for eid in candidate_negs:
            examples.append({"cid": cid, "eid": eid, "label": 0.0})

    rng.shuffle(examples)
    return examples


reranker_train_examples = build_reranker_examples(train_claims, train_bm25_candidates)
pos = sum(1 for x in reranker_train_examples if x["label"] == 1.0)
neg = len(reranker_train_examples) - pos
print(
    f"Reranker training examples: {len(reranker_train_examples):,} | positives={pos:,} negatives={neg:,} neg/pos={neg / max(pos, 1):.2f}"
    )

In [ ]:
class RerankerPairDataset(Dataset):
    def __init__(self, examples, claims_dict, evidence_dict, tokenizer, max_len=RERANKER_MAX_LEN):
        self.examples = examples
        self.claims = claims_dict
        self.evidence = evidence_dict
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        cid = ex["cid"]
        eid = ex["eid"]
        enc = self.tokenizer(
            self.claims[cid]["claim_text"],
            self.evidence[eid],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(ex["label"], dtype=torch.float32)
        return item


print("Loading reranker:", RERANKER_MODEL_NAME)
reranker_tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL_NAME)
reranker_model = AutoModelForSequenceClassification.from_pretrained(
    RERANKER_MODEL_NAME,
    num_labels=1,
    ignore_mismatched_sizes=True,
).to(DEVICE)

reranker_ds = RerankerPairDataset(reranker_train_examples, train_claims, evidence, reranker_tokenizer)
reranker_loader = DataLoader(reranker_ds, batch_size=RERANKER_BATCH_SIZE, shuffle=True)

# For reranking, controlled negative sampling already defines the training balance.
# Extra pos_weight often over-corrects and hurts top-K ranking.
reranker_loss_fn = nn.BCEWithLogitsLoss()
print("Reranker loss: BCEWithLogitsLoss without pos_weight")

In [ ]:
@torch.no_grad()
def score_candidates_with_reranker(claims_dict, bm25_candidates, split_name="dev"):
    reranker_model.eval()
    score_cache = {}
    all_items = list(claims_dict.items())

    print(f"Scoring {split_name} candidates with cross-encoder reranker...")
    for cid, claim in tqdm(all_items):
        cand_pairs = bm25_candidates[cid]
        cand_eids = [eid for eid, _ in cand_pairs]
        bm25_scores = np.array([score for _, score in cand_pairs], dtype=np.float32)
        logits_all = []

        for start in range(0, len(cand_eids), RERANKER_EVAL_BATCH_SIZE):
            batch_eids = cand_eids[start:start + RERANKER_EVAL_BATCH_SIZE]
            claims_batch = [claim["claim_text"]] * len(batch_eids)
            ev_batch = [evidence[eid] for eid in batch_eids]
            enc = reranker_tokenizer(
                claims_batch,
                ev_batch,
                truncation=True,
                padding=True,
                max_length=RERANKER_MAX_LEN,
                return_tensors="pt",
            ).to(DEVICE)
            logits = reranker_model(**enc).logits.squeeze(-1)
            logits_all.extend(logits.detach().cpu().tolist())

        logits_np = np.array(logits_all, dtype=np.float32)
        score_cache[cid] = {
            "eids": cand_eids,
            "bm25": bm25_scores,
            "ce_logit": logits_np,
            "ce_prob": 1.0 / (1.0 + np.exp(-logits_np)),
        }
    return score_cache


def select_fixed_k_ce(score_cache, k):
    out = {}
    for cid, entry in score_cache.items():
        order = np.argsort(-entry["ce_logit"])[:k]
        out[cid] = [entry["eids"][int(i)] for i in order]
    return out


def select_dynamic_threshold_ce(score_cache, threshold, max_k=MAX_FINAL_K, min_k=MIN_FINAL_K):
    out = {}
    for cid, entry in score_cache.items():
        probs = entry["ce_prob"]
        order = np.argsort(-probs)
        selected = [int(i) for i in order[:max_k] if probs[int(i)] >= threshold]
        if len(selected) < min_k:
            selected = [int(i) for i in order[:min_k]]
        out[cid] = [entry["eids"][i] for i in selected[:max_k]]
    return out


def select_relative_logit_ce(score_cache, delta, max_k=MAX_FINAL_K, min_k=MIN_FINAL_K):
    """Dynamic K by keeping candidates whose logit is within delta of the top candidate."""
    out = {}
    for cid, entry in score_cache.items():
        logits = entry["ce_logit"]
        order = np.argsort(-logits)
        top = float(logits[int(order[0])])
        selected = [int(i) for i in order[:max_k] if top - float(logits[int(i)]) <= delta]
        if len(selected) < min_k:
            selected = [int(i) for i in order[:min_k]]
        out[cid] = [entry["eids"][i] for i in selected[:max_k]]
    return out


def tune_retrieval_from_ce_scores(dev_score_cache):
    rows = []
    for k in FIXED_K_GRID:
        retr = select_fixed_k_ce(dev_score_cache, k=k)
        rows.append(
            {
                "mode": "fixed_k",
                "k": k,
                "threshold": np.nan,
                "delta": np.nan,
                "retrieval_F": evaluate_retrieval_only(retr, dev_claims),
                "avg_pred_evidence": np.mean([len(v) for v in retr.values()]),
            }
        )
    for threshold in THRESHOLD_GRID:
        retr = select_dynamic_threshold_ce(dev_score_cache, threshold=threshold, max_k=MAX_FINAL_K)
        rows.append(
            {
                "mode": "dynamic_threshold",
                "k": np.nan,
                "threshold": threshold,
                "delta": np.nan,
                "retrieval_F": evaluate_retrieval_only(retr, dev_claims),
                "avg_pred_evidence": np.mean([len(v) for v in retr.values()]),
            }
        )
    for delta in RELATIVE_LOGIT_DELTA_GRID:
        retr = select_relative_logit_ce(dev_score_cache, delta=delta, max_k=MAX_FINAL_K)
        rows.append(
            {
                "mode": "relative_logit",
                "k": np.nan,
                "threshold": np.nan,
                "delta": delta,
                "retrieval_F": evaluate_retrieval_only(retr, dev_claims),
                "avg_pred_evidence": np.mean([len(v) for v in retr.values()]),
            }
        )
    return pd.DataFrame(rows).sort_values("retrieval_F", ascending=False).reset_index(drop=True)


def choose_final_retrieval_setting(results_df):
    """Choose the final selector from dev results with an explicit, reportable policy."""
    results_df = results_df.sort_values("retrieval_F", ascending=False).reset_index(drop=True)
    best = results_df.iloc[0]

    if FINAL_RETRIEVAL_POLICY == "best_dev":
        chosen = best
    elif FINAL_RETRIEVAL_POLICY in {"fixed_k", "dynamic_threshold", "relative_logit"}:
        family = results_df[results_df["mode"] == FINAL_RETRIEVAL_POLICY]
        if family.empty:
            raise ValueError(f"No retrieval rows for policy {FINAL_RETRIEVAL_POLICY}")
        chosen = family.iloc[0]
    elif FINAL_RETRIEVAL_POLICY == "prefer_dynamic":
        dynamic = results_df[results_df["mode"].isin(["dynamic_threshold", "relative_logit"])]
        eligible = dynamic[dynamic["retrieval_F"] >= float(best["retrieval_F"]) - DYNAMIC_RETRIEVAL_TOLERANCE]
        chosen = eligible.iloc[0] if not eligible.empty else best
    else:
        raise ValueError(f"Unknown FINAL_RETRIEVAL_POLICY: {FINAL_RETRIEVAL_POLICY}")

    chosen = chosen.to_dict()
    print("Final retrieval policy:", FINAL_RETRIEVAL_POLICY)
    print("Best dev row:", best.to_dict())
    print("Chosen row:", chosen)
    return chosen


def apply_retrieval_setting(score_cache, row):
    mode = row["mode"]
    if mode == "fixed_k":
        return select_fixed_k_ce(score_cache, k=int(row["k"]))
    if mode == "dynamic_threshold":
        return select_dynamic_threshold_ce(score_cache, threshold=float(row["threshold"]), max_k=MAX_FINAL_K)
    if mode == "relative_logit":
        return select_relative_logit_ce(score_cache, delta=float(row["delta"]), max_k=MAX_FINAL_K)
    raise ValueError(f"Unknown retrieval mode: {mode}")


In [ ]:
# Train cross-encoder reranker. Dev retrieval F is evaluated after each epoch.
reranker_optimizer = torch.optim.AdamW(reranker_model.parameters(), lr=RERANKER_LR, weight_decay=WEIGHT_DECAY)
reranker_total_steps = len(reranker_loader) * RERANKER_EPOCHS
reranker_scheduler = get_linear_schedule_with_warmup(
    reranker_optimizer,
    num_warmup_steps=int(0.1 * reranker_total_steps),
    num_training_steps=reranker_total_steps,
)

best_reranker_state = None
best_reranker_row = None
best_reranker_F = -1.0

for epoch in range(1, RERANKER_EPOCHS + 1):
    reranker_model.train()
    losses = []
    t0 = time.time()

    for batch in tqdm(reranker_loader, desc=f"Reranker epoch {epoch}/{RERANKER_EPOCHS}"):
        labels = batch.pop("labels").to(DEVICE)
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = reranker_model(**batch).logits.squeeze(-1)
        loss = reranker_loss_fn(logits, labels)

        reranker_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(reranker_model.parameters(), 1.0)
        reranker_optimizer.step()
        reranker_scheduler.step()
        losses.append(float(loss.item()))

    # Evaluate reranker retrieval quality on dev by sweeping fixed K and dynamic selectors.
    dev_ce_scores_tmp = score_candidates_with_reranker(dev_claims, dev_bm25_candidates, split_name=f"dev_epoch{epoch}")
    retrieval_results_tmp = tune_retrieval_from_ce_scores(dev_ce_scores_tmp)
    row = choose_final_retrieval_setting(retrieval_results_tmp)
    print(
        f"Epoch {epoch}: loss={np.mean(losses):.4f} | "
        f"chosen dev retrieval_F={row['retrieval_F']:.4f} | setting={row} | time={time.time() - t0:.1f}s"
    )

    if row["retrieval_F"] > best_reranker_F:
        best_reranker_F = float(row["retrieval_F"])
        best_reranker_row = row
        best_reranker_state = {k: v.detach().cpu().clone() for k, v in reranker_model.state_dict().items()}

print("Best reranker retrieval setting under final policy:")
print(best_reranker_row)
if best_reranker_state is not None:
    reranker_model.load_state_dict(best_reranker_state)


In [ ]:
# Final retrieval caches after loading the best reranker state.
dev_ce_scores = score_candidates_with_reranker(dev_claims, dev_bm25_candidates, split_name="dev_final")
retrieval_results = tune_retrieval_from_ce_scores(dev_ce_scores)
display(retrieval_results.head(30))

best_row = choose_final_retrieval_setting(retrieval_results)

dev_retrieval = apply_retrieval_setting(dev_ce_scores, best_row)
validate_retrieval_coverage(dev_claims, dev_retrieval, split_name="dev")

# Score train candidates as well; classifier will be trained on retrieved evidence.
train_ce_scores = score_candidates_with_reranker(train_claims, train_bm25_candidates, split_name="train_final")
train_retrieval = apply_retrieval_setting(train_ce_scores, best_row)
validate_retrieval_coverage(train_claims, train_retrieval, split_name="train")

majority = majority_label(train_claims)
print("Majority label:", majority)
dev_majority_predictions = build_predictions(dev_claims, dev_retrieval, default_label=majority)
print("\nDev score with cross-encoder retrieval + majority label:")
retrieval_majority_metrics = evaluate_submission(dev_majority_predictions, dev_claims)
write_predictions(dev_majority_predictions, OUTPUT_DIR / "dev-cross-encoder-retrieval-majority.json")


# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## MiniLM classifier over retrieved evidence

This keeps the teammate's simple joint classifier: concatenate retrieved evidence passages and fine-tune a lightweight Transformer classifier. The key improvement is that retrieval is now produced by a supervised cross-encoder reranker, and classifier training uses retrieved evidence rather than oracle evidence.

In [ ]:
class ClaimEvidenceDataset(Dataset):
    def __init__(self, claims_dict, evidence_dict, retrieval_dict, tokenizer, max_len=MAX_SEQ_LEN):
        self.cids = list(claims_dict.keys())
        self.claims = claims_dict
        self.evidence = evidence_dict
        self.retrieval = retrieval_dict
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.has_label = "claim_label" in next(iter(claims_dict.values()))

    def __len__(self):
        return len(self.cids)

    def __getitem__(self, idx):
        cid = self.cids[idx]
        claim = self.claims[cid]
        ev_ids = self.retrieval[cid]
        # Explicit [SEP] between evidence passages helps preserve evidence boundaries.
        sep = f" {self.tokenizer.sep_token} "
        ev_texts = [self.evidence[eid] for eid in ev_ids if eid in self.evidence]
        if not ev_texts:
            ev_texts = [next(iter(self.evidence.values()))]
        ev_text = sep.join(ev_texts)

        enc = self.tokenizer(
            claim["claim_text"],
            ev_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.has_label:
            item["labels"] = torch.tensor(LABEL2ID[claim["claim_label"]], dtype=torch.long)
        return item


def make_loader(claims_dict, retrieval_dict, tokenizer, batch_size, shuffle=False, drop_last=False):
    validate_retrieval_coverage(claims_dict, retrieval_dict, split_name="loader")
    ds = ClaimEvidenceDataset(claims_dict, evidence, retrieval_dict, tokenizer, max_len=MAX_SEQ_LEN)
    return ds, DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)


In [ ]:
print("Loading classifier:", CLASSIFIER_MODEL_NAME)
classifier_tokenizer = AutoTokenizer.from_pretrained(CLASSIFIER_MODEL_NAME)
classifier_model = AutoModelForSequenceClassification.from_pretrained(
    CLASSIFIER_MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
).to(DEVICE)

n_params = sum(p.numel() for p in classifier_model.parameters())
print(f"Classifier parameters: {n_params:,}")

train_ds, train_loader = make_loader(
    train_claims, train_retrieval, classifier_tokenizer, TRAIN_BATCH_SIZE, shuffle=True
    )
dev_ds, dev_loader = make_loader(dev_claims, dev_retrieval, classifier_tokenizer, EVAL_BATCH_SIZE, shuffle=False)

label_counts = Counter(c["claim_label"] for c in train_claims.values())
raw_class_weights = torch.tensor(
    [len(train_claims) / (len(LABELS) * label_counts[label]) for label in LABELS],
    dtype=torch.float32,
    device=DEVICE,
)
class_weights = raw_class_weights if CLASSIFIER_USE_CLASS_WEIGHTS else None
print("Raw class weights:", dict(zip(LABELS, raw_class_weights.detach().cpu().tolist())))
print("Using class weights:", CLASSIFIER_USE_CLASS_WEIGHTS)

In [ ]:
def predict_labels(model, loader, dataset):
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
            logits = model(**batch).logits
            preds.extend(logits.argmax(dim=-1).detach().cpu().tolist())
    return {dataset.cids[i]: ID2LABEL[p] for i, p in enumerate(preds)}


def evaluate_classifier_on_dev(model):
    pred_labels = predict_labels(model, dev_loader, dev_ds)
    predictions = build_predictions(dev_claims, dev_retrieval, label_predictions=pred_labels)
    return evaluate_submission(predictions, dev_claims, verbose=False), pred_labels, predictions

In [ ]:
classifier_optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=CLASSIFIER_LR, weight_decay=WEIGHT_DECAY)
classifier_total_steps = len(train_loader) * CLASSIFIER_EPOCHS
classifier_scheduler = get_linear_schedule_with_warmup(
    classifier_optimizer,
    num_warmup_steps=int(0.1 * classifier_total_steps),
    num_training_steps=classifier_total_steps,
)
classifier_loss_fn = nn.CrossEntropyLoss(weight=class_weights)

best_classifier_state = None
best_metrics = {"F": 0.0, "A": 0.0, "H": -1.0}

for epoch in range(1, CLASSIFIER_EPOCHS + 1):
    classifier_model.train()
    losses = []
    t0 = time.time()

    for batch in tqdm(train_loader, desc=f"Classifier epoch {epoch}/{CLASSIFIER_EPOCHS}"):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        labels = batch.pop("labels")

        logits = classifier_model(**batch).logits
        loss = classifier_loss_fn(logits, labels)

        classifier_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(classifier_model.parameters(), 1.0)
        classifier_optimizer.step()
        classifier_scheduler.step()

        losses.append(float(loss.item()))

    metrics, _, _ = evaluate_classifier_on_dev(classifier_model)
    print(
        f"Epoch {epoch}: loss={np.mean(losses):.4f} | "
        f"dev F={metrics['F']:.4f} A={metrics['A']:.4f} H={metrics['H']:.4f} | "
        f"time={time.time() - t0:.1f}s"
    )

    # Retrieval is fixed during classifier training, so H is the right selection metric here.
    if metrics["H"] > best_metrics["H"]:
        best_metrics = metrics
        best_classifier_state = {k: v.detach().cpu().clone() for k, v in classifier_model.state_dict().items()}

print("Best dev metrics:", best_metrics)
if best_classifier_state is not None:
    classifier_model.load_state_dict(best_classifier_state)


# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [ ]:
# Final dev prediction and official-format output.
dev_pred_labels = predict_labels(classifier_model, dev_loader, dev_ds)
dev_predictions = build_predictions(dev_claims, dev_retrieval, label_predictions=dev_pred_labels)
print("Final dev score:")
final_dev_metrics = evaluate_submission(dev_predictions, dev_claims)
write_predictions(dev_predictions, OUTPUT_DIR / "dev-predictions.json")

In [ ]:
# Optional: run the official eval.py if it is available in the current directory.
import subprocess
import sys

eval_py = Path("eval.py")
if eval_py.exists():
    cmd = [
        sys.executable, "eval.py", "--predictions", str(OUTPUT_DIR / "dev-predictions.json"), "--groundtruth",
        str(DATA_DIR / "dev-claims.json")
    ]
    print("Running:", " ".join(cmd))
    completed = subprocess.run(cmd, text=True, capture_output=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
else:
    print("eval.py not found; skipped official evaluator subprocess.")

In [ ]:
# Simple error analysis for the report.
def confusion_matrix_df(gold_claims, pred_labels):
    mat = pd.DataFrame(0, index=LABELS, columns=LABELS)
    for cid, claim in gold_claims.items():
        mat.loc[claim["claim_label"], pred_labels[cid]] += 1
    return mat


cm = confusion_matrix_df(dev_claims, dev_pred_labels)
print("Confusion matrix: rows=gold, cols=pred")
display(cm)

per_class_rows = []
for label in LABELS:
    cids = [cid for cid, c in dev_claims.items() if c["claim_label"] == label]
    acc = np.mean([dev_pred_labels[cid] == label for cid in cids]) if cids else 0.0
    retr_f = np.mean(
        [evidence_f1_for_claim(dev_retrieval[cid], dev_claims[cid]["evidences"]) for cid in cids]
        ) if cids else 0.0
    per_class_rows.append({"label": label, "n": len(cids), "class_acc": acc, "retrieval_F": retr_f})

per_class = pd.DataFrame(per_class_rows)
display(per_class)

In [ ]:
# Generate test predictions for leaderboard / final submission.
# The test set is unlabeled. Do not inspect or manually modify predictions.

test_bm25_candidates = compute_or_load_bm25_candidates(test_claims, "test", BM25_CANDIDATE_K)
test_ce_scores = score_candidates_with_reranker(test_claims, test_bm25_candidates, split_name="test_final")
test_retrieval = apply_retrieval_setting(test_ce_scores, best_row)
validate_retrieval_coverage(test_claims, test_retrieval, split_name="test")

test_ds, test_loader = make_loader(
    test_claims,
    test_retrieval,
    classifier_tokenizer,
    EVAL_BATCH_SIZE,
    shuffle=False,
    drop_last=False,
)
test_pred_labels = predict_labels(classifier_model, test_loader, test_ds)
test_predictions = build_predictions(test_claims, test_retrieval, label_predictions=test_pred_labels)

# Keep the generic name expected by many leaderboard/upload workflows.
write_predictions(test_predictions, OUTPUT_DIR / "test-output.json")
print("Test output ready:", OUTPUT_DIR / "test-output.json")
print("Final retrieval setting used for test:", best_row)


## Optional final-training note

For the final leaderboard submission, it is allowed to train on train+dev after all hyperparameters have been selected on dev. This notebook keeps the safer report setting: train on train, tune/evaluate on dev, then predict test. If the team decides to use train+dev, add one final rerun cell that merges train and dev only after recording dev results.

## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed.*

This notebook uses two lightweight dataset classes:

- `RerankerPairDataset` for cross-encoder relevance fine-tuning.
- `ClaimEvidenceDataset` for final DistilBERT claim classification.